In [1]:
pip install tifffile imagecodecs

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""Précalcule le raster GEM (PGA à 475 ans, v2023.1) en un petit fichier ``gem_pga475_compact.npz`` à déposer dans data/seisme/ du dépôt.

Script autonome (aucune dépendance au projet) : il faut numpy et tifffile (``pip install tifffile imagecodecs``). Utilisable en ligne de commande ou dans
Jupyter/Spyder.

Usage :
    python build_gem_compact.py                    -> cherche dans le dossier courant (ou celui du script) un .tif dont le nom contient « pga »,
                                                      ou un .zip qui en contient un, et écrit gem_pga475_compact.npz au même endroit
    python build_gem_compact.py DOSSIER_OU_FICHIER [SORTIE.npz]

Licence des données GEM : CC BY-NC-SA 4.0 (usage non commercial, partage dans les mêmes conditions, avec mention de la source : GEM Global Seismic
Hazard Map v2023.1). Mémoire nécessaire : environ 1 Go.
"""
from __future__ import annotations

import math
import sys
import tempfile
import zipfile
from pathlib import Path
from typing import Optional

import numpy as np

_NODATA_CODE = 65535


class SeismicError(Exception):
    pass


class TileError(Exception):
    pass


def _georef(page) -> tuple[float, float, float, float]:
    """(x0, y0, sx, sy) : coin haut-gauche et taille de pixel (sy > 0 vers le bas) lus dans les tags GeoTIFF."""
    tag = page.tags.get
    if tag(34264) is not None:                               # ModelTransformation
        m = tag(34264).value
        return float(m[3]), float(m[7]), float(m[0]), -float(m[5])
    if tag(33550) is not None and tag(33922) is not None:
        scale, tie = tag(33550).value, tag(33922).value
        return (float(tie[3]) - float(tie[0]) * float(scale[0]), float(tie[4]) + float(tie[1]) * float(scale[1]),
                float(scale[0]), float(scale[1]))
    raise TileError("géoréférencement absent du fichier (tags GeoTIFF)")


def _nodata(page) -> Optional[float]:
    nd = page.tags.get(42113).value if page.tags.get(42113) is not None else None
    try:
        return float(str(nd).strip("\x00 ")) if nd is not None else None
    except ValueError:
        return None


def read_full(tif: Path) -> tuple:
    """Lit tout le raster : ``(tableau float32 (lignes, colonnes) avec NaN pour sans-donnée, (x0, y0), (sx, sy))``."""
    import tifffile
    with tifffile.TiffFile(str(tif)) as tf:
        page = tf.pages[0]
        H, W = int(page.imagelength), int(page.imagewidth)
        x0, y0, sx, sy = _georef(page)
        nd = _nodata(page)
        try:
            arr = np.squeeze(page.asarray())
            if arr.ndim == 1 and arr.size == H * W:
                arr = arr.reshape(H, W)
        except Exception:                                       # noqa: BLE001 - repli : lecture bande par bande
            arr = None
        if arr is None or arr.shape != (H, W):
            if page.is_tiled:
                raise SeismicError("lecture du raster impossible (organisation en tuiles non lisible par tifffile).")
            rps_ = int(page.rowsperstrip)
            arr = np.full((H, W), np.nan, np.float32)
            fh = tf.filehandle
            for seg in range(len(page.dataoffsets)):
                if not page.databytecounts[seg]:
                    continue
                fh.seek(page.dataoffsets[seg])
                data, _idx, _ = page.decode(fh.read(page.databytecounts[seg]), seg)
                blk = np.squeeze(data)
                blk = blk.reshape(-1, W) if blk.ndim == 1 else blk
                arr[seg * rps_: seg * rps_ + blk.shape[0]] = blk[: H - seg * rps_]
        arr = arr.astype(np.float32)
        if nd is not None:
            arr[arr == nd] = np.nan
        arr[~np.isfinite(arr) | (arr < 0)] = np.nan
    return arr, (x0, y0), (sx, sy)


def build_compact(tif: Path, out_path: Path, max_mb: float = 90.0, progress=print) -> dict:
    """Convertit le raster GEM en fichier précalculé (PGA × ``scale`` en entiers 16 bits, 65535 = sans donnée)."""
    arr, origin, pixel = read_full(Path(tif))
    top = float(np.nanmax(arr)) if np.isfinite(arr).any() else 0.0
    if top <= 0:
        raise SeismicError("raster vide : aucune valeur de PGA positive.")
    scale = float(10 ** math.floor(math.log10(65000.0 / top)))
    q = np.where(np.isfinite(arr), np.clip(np.rint(arr * scale), 0, 65534), _NODATA_CODE).astype(np.uint16)
    np.savez_compressed(out_path, pga=q, origin=np.array(origin, np.float64), pixel=np.array(pixel, np.float64), scale=np.float32(scale))
    mb = Path(out_path).stat().st_size / 1e6
    progress(f"écrit : {out_path} ({mb:.1f} Mo) ; {q.shape[1]}×{q.shape[0]} pixels, PGA max {top:.3f} g, précision {1 / scale:g} g")
    if mb > max_mb:
        raise SeismicError(f"fichier précalculé trop gros ({mb:.0f} Mo > {max_mb:g} Mo).")
    return {"pixels": [int(q.shape[1]), int(q.shape[0])], "taille_mo": round(mb, 2), "pga_max": round(top, 4), "precision_g": 1.0 / scale}


def find_tif(src: Path, tmp: Path) -> Path:
    """Fichier .tif GEM : le chemin donné, sinon un .tif « pga » du dossier, sinon extrait d'un .zip du dossier."""
    src = Path(src)
    if src.is_file():
        if src.suffix.lower() == ".zip":
            return _from_zip(src, tmp)
        return src
    tifs = sorted(p for p in src.iterdir() if p.is_file() and p.suffix.lower() in (".tif", ".tiff") and "pga" in p.name.lower())
    if tifs:
        return tifs[0]
    for z in sorted(src.glob("*.zip")):
        if zipfile.is_zipfile(z):
            try:
                return _from_zip(z, tmp)
            except SeismicError:
                continue
    raise SeismicError(f"aucun raster GEM (.tif contenant « pga », ou .zip qui en contient un) dans {src}")


def _from_zip(z: Path, tmp: Path) -> Path:
    with zipfile.ZipFile(z) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith((".tif", ".tiff"))]
        if not names:
            raise SeismicError(f"aucun .tif dans {z.name}")
        dest = Path(tmp) / Path(names[0]).name
        with zf.open(names[0]) as a, open(dest, "wb") as b:
            while True:
                buf = a.read(1 << 20)
                if not buf:
                    break
                b.write(buf)
    return dest


def main(argv=None):
    argv = [a for a in (sys.argv[1:] if argv is None else argv) if not a.startswith("-") and not a.endswith(".json")]
    try:
        here = Path(__file__).resolve().parent
    except NameError:                                           # notebook
        here = Path.cwd()
    src = Path(argv[0]) if argv else here
    if not src.exists():
        print(f"Introuvable : {src}")
        return None
    out = Path(argv[1]) if len(argv) > 1 else (src if src.is_dir() else src.parent) / "gem_pga475_compact.npz"
    try:
        with tempfile.TemporaryDirectory() as tmp:
            tif = find_tif(src, Path(tmp))
            print(f"Raster : {tif}\nSortie : {out}")
            res = build_compact(tif, out)
    except (SeismicError, TileError) as e:
        print(f"ERREUR : {e}")
        return None
    except ImportError as e:
        print(f"ERREUR : bibliothèque manquante ({e}). Installez : pip install tifffile imagecodecs")
        return None
    print(res)
    return res


if __name__ == "__main__":
    main()

Raster : C:\Users\maxime.trevisani_squ\Desktop\Sismic_data\v2023_1_pga_475_rock_3min.tif
Sortie : C:\Users\maxime.trevisani_squ\Desktop\Sismic_data\gem_pga475_compact.npz
écrit : C:\Users\maxime.trevisani_squ\Desktop\Sismic_data\gem_pga475_compact.npz (5.9 Mo) ; 7200×3000 pixels, PGA max 1.893 g, précision 0.0001 g
{'pixels': [7200, 3000], 'taille_mo': 5.9, 'pga_max': 1.893, 'precision_g': 0.0001}
